### Question 1

**TF-IDF**

In [32]:
import numpy as np
import re
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [33]:
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

In [34]:
# Text Preprocessing
def preprocess(text):
    text = text.lower()                      # lowercase
    text = re.sub(r'[^a-z\s]', '', text)     # remove punctuation
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]  # remove stopwords
    return " ".join(words)

cleaned_dataset = [preprocess(doc) for doc in dataset]

In [35]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_dataset)

In [36]:
k = 2
km = KMeans(n_clusters=k)
km.fit(X)

y_pred = km.predict(X)

In [37]:
table_data = [["Document", "Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(cleaned_dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))

Document                               Cluster
-----------------------------------  ---------
love playing football weekends               1
enjoy hiking camping mountains               1
like read books watch movies                 0
prefer playing video games sports            1
love listening music going concerts          1


In [38]:
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples

print("Purity:", purity)

Purity: 0.8


**Word2Vec**

In [24]:
import numpy as np
import re
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [25]:
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

In [26]:
# Preprocessing
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return words   # return list for Word2Vec


In [27]:
tokenized_dataset = [preprocess(doc) for doc in dataset]
word2vec_model = Word2Vec(
    sentences=tokenized_dataset,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [28]:
X = np.array([
    np.mean(
        [word2vec_model.wv[word] for word in doc if word in word2vec_model.wv],
        axis=0
    )
    for doc in tokenized_dataset
])

In [29]:
k = 2
km = KMeans(n_clusters=k)
km.fit(X)

y_pred = km.predict(X)

C:\Users\nabie\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [30]:
table_data = [["Document", "Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(tokenized_dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))

Document                                               Cluster
---------------------------------------------------  ---------
['love', 'playing', 'football', 'weekends']                  1
['enjoy', 'hiking', 'camping', 'mountains']                  1
['like', 'read', 'books', 'watch', 'movies']                 0
['prefer', 'playing', 'video', 'games', 'sports']            1
['love', 'listening', 'music', 'going', 'concerts']          1


In [39]:
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples

print("Purity:", purity)

Purity: 0.8


***Yes, purity usually changes (often improves). Because text preprocessing removes noise (stopwords like "the", "and"), standardizes words (lowercase) and keeps only meaningful words. The result clusters become more accurate, similar documents group better, and purity often increases***